# QLoRA Fine-tune — Data Prep Verification

Verify `data/train.jsonl` before training on the server: category balance, and that each
example renders to a clean SFT string via the Llama-3.1 chat template.

The fine-tune teaches **behavioral policy** (not facts): when to ask vs. answer, which tool
to call, and the advice-boundary. Three categories:
1. `tool_selection` — numeric question + full info → emit the right tool-call JSON
2. `context_elicitation` — missing a personal value → ask, don't guess
3. `advice_boundary` — "which product should I buy" → redirect to principles, no product

In [ ]:
import json
from collections import Counter

examples = [json.loads(l) for l in open('../data/train.jsonl')]
print(f'{len(examples)} examples')
print('Category balance:', Counter(e['category'] for e in examples))

## Render each category through the real chat template

Uses the ungated `NousResearch` mirror locally (identical tokenizer/template to the gated
`meta-llama/Llama-3.1-8B-Instruct`). `format_example` trims any trailing generation prompt
so the SFT target ends cleanly on the assistant turn.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('NousResearch/Meta-Llama-3.1-8B-Instruct')

def format_example(example, tokenizer):
    text = tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)
    eot = '<|eot_id|>'
    if eot in text:
        text = text[: text.rfind(eot) + len(eot)]
    return text

for cat in ['tool_selection', 'context_elicitation', 'advice_boundary']:
    ex = next(e for e in examples if e['category'] == cat)
    out = format_example(ex, tok)
    assert out.endswith('<|eot_id|>'), 'target does not end cleanly'
    target = out.rsplit('<|end_header_id|>', 1)[-1].replace('<|eot_id|>', '').strip()
    print(f'=== {cat} ===')
    print('USER:  ', ex['messages'][1]['content'])
    print('TARGET:', target[:300])
    print()

## LoRA config (for reference — actual training runs via `src/train_qlora.py` on the server)

- 4-bit NF4 quantized base (`load_in_4bit`, `bnb_4bit_quant_type='nf4'`, double-quant)
- LoRA: `r=16`, `lora_alpha=32`, `target_modules=['q_proj','v_proj']`, `lora_dropout=0.1`
- 3 epochs, batch 4 × grad-accum 2, lr 2e-4 cosine, checkpoint every 50 steps

Do NOT train here (no CUDA on the Mac). Run on the IEOR A5000 server — see the handoff notes.